<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/yolo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLVS3-HST — YOLOv8s-Pose training

Training on the HST sample dataset with 37 keypoints. The training output from the original Colab run is kept below.

The dataset uses `images/{train,val,test}` and `labels/{train,val,test}`. Select a GPU runtime if running the notebook again.

In [ ]:
!pip -q install -U ultralytics pyyaml


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# The converted dataset is stored in Drive.
DRIVE_DATASET = Path('/content/drive/MyDrive/hst_yolo_pose')
LOCAL_DATASET = Path('/content/hst_yolo_pose')

assert DRIVE_DATASET.is_dir(), f'Dataset folder not found: {DRIVE_DATASET}'
assert (DRIVE_DATASET / 'images' / 'train').is_dir(), 'Expected images/train in the original folder structure'
assert (DRIVE_DATASET / 'labels' / 'train').is_dir(), 'Expected labels/train in the original folder structure'

# Copy images locally to avoid slow reads during training.
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET, dirs_exist_ok=True)
print('Copied dataset to:', LOCAL_DATASET)


Mounted at /content/drive
Copied dataset to: /content/hst_yolo_pose


In [ ]:
import torch
import yaml
from pathlib import Path
from ultralytics import YOLO

yaml_path = Path("/content/hst_yolo_pose/data.yaml")

with open(yaml_path) as f:
    data = yaml.safe_load(f)

data["path"] = "/content/hst_yolo_pose"
data["kpt_shape"] = [37, 3]

# Use the experimental OKS sigma (0.10) from this run.
data["kpt_oks_sigmas"] = [0.10] * 37

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

model = YOLO("yolov8s-pose.pt")

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=25,
    seed=42,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    project="/content/drive/MyDrive/HST_YOLO_Runs",
    name="hst_pose_sigma_010",
    save=True,
    plots=True,
)


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hst_yolo_pose/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=hst_pose_sigma_010, nbs=64, nms=None, opset=None, 